# 06d — Diagnósticos comparados: universo completo (2003T3-2025T4) contra 2016T2-2025T4

Repite las secciones de `06b_diagnosticos_trayectoria.ipynb` (tendencia por trimestre,
regresión a la media por decil, conteos train/holdout, salida de la pobreza) para dos
universos: **(a)** 2003T3-2025T4 completo y **(b)** 2016T2-2025T4, con el mismo esquema
train/holdout (`panel.particion_train_holdout`: holdout = origen `>= 2023T1`, train = destino
`<= 2022T4`, embargo de 1 trimestre). Solo descriptivo: escribe `data/meta/dos_universos_*.csv`.

**Criterio principal: `par_valido_c`** (variante C, decisión sobre `05d`: ningún par con una
punta con ingresos imputados por INDEC). La variante A (`par_valido`, imputados adentro) se
repite como sensibilidad en la línea base.

**Advertencias**
- `variacion_comun_*` (de `06`/`06c`) se calcula sobre la misma ventana `t→t+h`: usa
  información de `t+h`, es descriptivo, nunca una feature.
- `PONDERA` (histórica) y `PONDIH` (regular) no son comparables: toda estadística que junta
  eras usa `peso_normalizado_trimestre` y se reporta también por era (`era` es obligatoria, ver
  `decisiones.md`).
- Los pares con cortes conocidos (intervención 2007-2015, ruptura de `CODUSU`, hueco 2007T3,
  `ITF==0` elevado, 2020T2 telefónico) se marcan, no se excluyen; la sensibilidad sin ellos va
  al final.

In [ ]:
import sys
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd

import cortes_trayectoria as ct
import imputacion_ingresos as ii
import linea_base_trayectoria as lb
import panel
import trayectoria_distancia
from armonizacion import trimestres_publicados

TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
META = REPO / "data" / "meta"
HOLDOUT_DESDE = (2023, 1)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)


def cargar_capa(h: int) -> pd.DataFrame:
    partes = [pd.read_parquet(p) for p in sorted((TRAYECTORIA / f"pares_h{h}").glob("ANO4=*/TRIMESTRE=*/data.parquet"))]
    return pd.concat(partes, ignore_index=True)


pares = {h: cargar_capa(h) for h in (1, 4)}
for h in (1, 4):
    print(f"h={h}: {len(pares[h]):,} pares --", pares[h]["era_t"].value_counts().to_dict())

## Columnas de imputación y de corte en los pares regulares (simetría entre eras)

Los pares regulares (`06`) no tienen columnas de imputación. Se completan en memoria con
`imputado_* = False`, `par_valido_c = par_valido` y `par_valido_log_c = par_valido_log`. Es
simétrico con la era histórica: en la regular la no respuesta (`DECCFR==12`, `PONDIH==0`) ya
tiene el ratio en `NA` desde `05` (`ingreso_no_declarado`), así que `par_valido` ya la excluye —
el "par sin ingreso no declarado en ninguna punta" de la regular es exactamente `par_valido`.

`par_valido*` puede venir en `NA` desde `construir_pares_distancia` cuando `identidad_confirmada`
es `NA` (jefe/a ambiguo) y la variación existe: se trata como no válido (`False`), mismo criterio
que `identidad_confirmada == True`; la cantidad se imprime abajo.

Los flags de corte se aplican también a la regular (`par_toca_2020t2` es el único que puede
dar `True` ahí), con el mismo conjunto de trimestres `ITF==0` elevado que `06c` registró.

In [ ]:
pct_itf0 = {}
for p in sorted(DISTANCIA.glob("ANO4=*/TRIMESTRE=*/data.parquet")):
    q = (int(p.parent.parent.name.split("=")[1]), int(p.parent.name.split("=")[1]))
    pct_itf0[q] = ct.pct_itf_cero_declarado(pd.read_parquet(p, columns=["ITF", "ingreso_no_declarado"]))
TRIMESTRES_ITF0 = ct.trimestres_itf0_elevado(pct_itf0, ct.UMBRAL_ITF0_PCT)
print(f"trimestres ITF==0 elevado (umbral {ct.UMBRAL_ITF0_PCT}%): {TRIMESTRES_ITF0}")

for h in (1, 4):
    df = pares[h]
    regular = (df["era_t"] == "regional_2016").to_numpy()
    df["era"] = df["era_t"]
    for lado in ("t", "th"):
        df[f"imputado_{lado}"] = df[f"imputado_{lado}"].astype("boolean")
        df.loc[regular, f"imputado_{lado}"] = False
    for columna in ("par_valido", "par_valido_log", "par_valido_c", "par_valido_log_c"):
        print(f"h={h}: {columna} NA -> False (solo pares históricos): {int(df.loc[~regular, columna].isna().sum())}")
        df[columna] = df[columna].astype("boolean").fillna(False).astype(bool)
    df.loc[regular, "par_valido_c"] = df.loc[regular, "par_valido"]
    df.loc[regular, "par_valido_log_c"] = df.loc[regular, "par_valido_log"]
    marcados = ct.marcar_cortes_par(df[regular], TRIMESTRES_ITF0)
    for flag in ct.FLAGS_CORTE:
        df[flag] = df[flag].astype("boolean")
        df.loc[regular, flag] = marcados[flag].to_numpy()
        df[flag] = df[flag].astype(bool)
    df["peso_hogar_era_t"] = ct.peso_hogar_era(df, "t")
    df["particion"] = panel.particion_train_holdout(df, HOLDOUT_DESDE)
    df["universo_b"] = df["ANO4_t"] >= 2016
    assert (df["era_t"] == df["era_th"]).all()
    pares[h] = df

UNIVERSOS = {"a_2003_2025": lambda df: pd.Series(True, index=df.index), "b_2016_2025": lambda df: df["universo_b"]}
CRITERIOS = {"C": ("par_valido_c", "par_valido_log_c"), "A": ("par_valido", "par_valido_log")}
for h in (1, 4):
    print(f"h={h}: flags de corte (n pares marcados):",
          {f: int(pares[h][f].sum()) for f in ct.FLAGS_CORTE})

## Tabla clave: N de pares válidos por universo × h × partición

In [ ]:
filas = []
for nombre_u, en_u in UNIVERSOS.items():
    for h in (1, 4):
        df = pares[h][en_u(pares[h])]
        for particion in ("train", "embargo", "holdout"):
            g = df[df["particion"] == particion]
            filas.append({"universo": nombre_u, "h": h, "particion": particion,
                          "n_par_valido_c": int(g["par_valido_c"].sum()), "n_par_valido": int(g["par_valido"].sum()),
                          "n_par_valido_log_c": int(g["par_valido_log_c"].sum()),
                          "n_par_valido_log": int(g["par_valido_log"].sum())})
conteos = pd.DataFrame(filas)
conteos.to_csv(META / "dos_universos_conteos.csv", index=False)
display(conteos.set_index(["universo", "h", "particion"]))

## 1. Tendencia por trimestre (`par_valido_c`)

Mediana ponderada (`peso_hogar_era_t`, dentro de cada trimestre — no mezcla ponderadores de
eras) de la variación simétrica y log, y dirección. El universo (b) es el subconjunto 2016+ de
la misma tabla.

In [ ]:
def tendencia_por_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for (anio, trimestre), g in df.groupby(["ANO4_t", "TRIMESTRE_t"]):
        g_log = g[g["par_valido_log_c"]]
        filas.append({"ANO4_t": anio, "TRIMESTRE_t": trimestre, "era": g["era"].iloc[0], "n_pares": len(g),
                      "mediana_variacion_simetrica": trayectoria_distancia.mediana_ponderada(g["variacion_simetrica"], g["peso_hogar_era_t"]),
                      "mediana_variacion_log": trayectoria_distancia.mediana_ponderada(g_log["variacion_log"], g_log["peso_hogar_era_t"]),
                      "pct_mejora": 100 * (g["direccion"] == "mejora").mean(),
                      "pct_empeora": 100 * (g["direccion"] == "empeora").mean()})
    return pd.DataFrame(filas)


for h in (1, 4):
    tendencia = tendencia_por_trimestre(pares[h][pares[h]["par_valido_c"]])
    tendencia.to_csv(META / f"dos_universos_tendencia_h{h}.csv", index=False)
    print(f"h={h}: resumen por era de la tendencia trimestral")
    display(tendencia.groupby("era")[["n_pares", "mediana_variacion_simetrica", "mediana_variacion_log",
                                      "pct_mejora", "pct_empeora"]].describe().T.round(3))

## 2. Regresión a la media por decil de `ratio_linea_pobreza_t` (`par_valido_c`)

Deciles sin ponderar, cortes propios de cada universo (mismo criterio que `06b`).

In [ ]:
filas = []
for nombre_u, en_u in UNIVERSOS.items():
    for h in (1, 4):
        df = pares[h][en_u(pares[h]) & pares[h]["par_valido_c"]].copy()
        df["decil"] = pd.qcut(df["ratio_linea_pobreza_t"].astype(float), 10, labels=False, duplicates="drop") + 1
        agregado = df.groupby("decil").agg(n=("variacion_simetrica", "size"),
                                           ratio_t_mediana=("ratio_linea_pobreza_t", "median"),
                                           variacion_simetrica_media=("variacion_simetrica", "mean"),
                                           variacion_log_media=("variacion_log", "mean")).reset_index()
        agregado["universo"], agregado["h"] = nombre_u, h
        filas.append(agregado)
regresion_decil = pd.concat(filas, ignore_index=True)
regresion_decil.to_csv(META / "dos_universos_regresion_media_decil.csv", index=False)
display(regresion_decil.pivot_table(index="decil", columns=["universo", "h"], values="variacion_simetrica_media").round(3))

## 3. Línea base: persistencia, media global y regresión a la media por decil

Entrenamiento con los pares `train` y evaluación en `holdout`, por universo × h × objetivo
(`variacion_simetrica` con `par_valido_c`, `variacion_log` con `par_valido_log_c`). Tres
predictores: **persistencia** (variación = 0), **media global** de train, y **regresión a la
media por decil** de `ratio_linea_pobreza_t` (cortes y medias de train,
`linea_base_trayectoria`). Métricas: MAE, RMSE y R² contra predecir la media de train; sin
ponderar y con `peso_normalizado_trimestre` (cada trimestre de origen pesa lo mismo). Se repite
con la variante A (`par_valido`/`par_valido_log`) como sensibilidad.

In [ ]:
OBJETIVOS = {"simetrica": ("variacion_simetrica", 0), "log": ("variacion_log", 1)}


def metricas_linea_base(df: pd.DataFrame, criterio: str, objetivo: str, filtro=None) -> list[dict]:
    columna_y, i_criterio = OBJETIVOS[objetivo]
    valido = df[CRITERIOS[criterio][i_criterio]]
    if filtro is not None:
        valido = valido & filtro
    train = df[valido & (df["particion"] == "train")]
    holdout = df[valido & (df["particion"] == "holdout")].copy()
    modelo = lb.ajustar_regresion_media_decil(train["ratio_linea_pobreza_t"], train[columna_y])
    holdout["_w"] = ct.peso_normalizado_trimestre(holdout.assign(_p=holdout["peso_hogar_era_t"]), "_p")
    predicciones = {"persistencia": pd.Series(0.0, index=holdout.index),
                    "media_global": pd.Series(modelo["media_global"], index=holdout.index),
                    "regresion_media_decil": lb.predecir_regresion_media_decil(modelo, holdout["ratio_linea_pobreza_t"])}
    filas = []
    for nombre, prediccion in predicciones.items():
        for ponderacion, peso in (("sin_ponderar", None), ("peso_normalizado_trimestre", holdout["_w"])):
            filas.append({"modelo": nombre, "ponderacion": ponderacion, "n_train": len(train),
                          **lb.metricas_prediccion(holdout[columna_y], prediccion, modelo["media_global"], peso)})
    return filas


filas = []
for criterio in ("C", "A"):
    for nombre_u, en_u in UNIVERSOS.items():
        for h in (1, 4):
            df = pares[h][en_u(pares[h])]
            for objetivo in OBJETIVOS:
                for fila in metricas_linea_base(df, criterio, objetivo):
                    filas.append({"criterio": criterio, "universo": nombre_u, "h": h, "objetivo": objetivo, **fila})
linea_base = pd.DataFrame(filas)
linea_base.to_csv(META / "dos_universos_linea_base.csv", index=False)
print("criterio principal C:")
display(linea_base[linea_base["criterio"] == "C"].set_index(["objetivo", "h", "ponderacion", "modelo", "universo"])
        [["n_train", "n", "mae", "rmse", "r2_vs_media_train"]].unstack("universo").round(4))

In [ ]:
print("sensibilidad, variante A:")
display(linea_base[linea_base["criterio"] == "A"].set_index(["objetivo", "h", "ponderacion", "modelo", "universo"])
        [["n_train", "n", "mae", "rmse", "r2_vs_media_train"]].unstack("universo").round(4))

## 4. Dispersión de la variable objetivo por era

In [ ]:
filas = []
for h in (1, 4):
    for criterio, (col_sim, col_log) in CRITERIOS.items():
        for era, g in pares[h][pares[h][col_sim]].groupby("era"):
            filas.append({"h": h, "criterio": criterio, "era": era, **ii.resumen_dispersion(g["variacion_simetrica"])})
dispersion_era = pd.DataFrame(filas)
dispersion_era.to_csv(META / "dos_universos_dispersion_era.csv", index=False)
display(dispersion_era.set_index(["h", "criterio", "era"]).round(4))

## 5. Sensibilidad sin los pares marcados

Línea base (regresión a la media por decil y persistencia, criterio C) quitando los pares
marcados: **(s1)** sin los flags puntuales (ruptura de `CODUSU` en el par o en la previa, hueco
2007T3 en el par o en la previa, `ITF==0` elevado, 2020T2); **(s2)** además sin el período
intervenido 2007-2015 — deja al universo (a) casi sin pares 2007-2015, por eso va aparte.

In [ ]:
filas = []
for nombre_s, flags in (("base", []), ("s1", ct.FLAGS_PUNTUALES), ("s2", ct.FLAGS_PUNTUALES + ["par_periodo_intervenido"])):
    for nombre_u, en_u in UNIVERSOS.items():
        for h in (1, 4):
            df = pares[h][en_u(pares[h])]
            sin_marca = ~df[flags].any(axis=1) if flags else pd.Series(True, index=df.index)
            for objetivo in OBJETIVOS:
                for fila in metricas_linea_base(df, "C", objetivo, filtro=sin_marca):
                    if fila["ponderacion"] == "sin_ponderar" and fila["modelo"] != "media_global":
                        filas.append({"sensibilidad": nombre_s, "universo": nombre_u, "h": h, "objetivo": objetivo, **fila})
sensibilidad = pd.DataFrame(filas)
sensibilidad.to_csv(META / "dos_universos_sensibilidad_marcados.csv", index=False)
display(sensibilidad.set_index(["objetivo", "h", "modelo", "universo", "sensibilidad"])
        [["n_train", "n", "mae", "rmse", "r2_vs_media_train"]].unstack("sensibilidad").round(4))

## 6. Efecto de composición de C a lo largo del tiempo

Por año de origen: p10/p50/p90 de `ratio_linea_pobreza_t` de los pares válidos bajo C contra
bajo A, y el % de pares válidos (A) que C excluye por imputación. La imputación sube de ~14%
de hogares (2006) a ~24% (2012), y los imputados se concentran en los quintiles altos (`05d`):
C corre la composición del universo histórico hacia abajo en la distribución.

In [ ]:
filas = []
for h in (1, 4):
    for anio, g in pares[h].groupby("ANO4_t"):
        bajo_a = g.loc[g["par_valido"], "ratio_linea_pobreza_t"].astype(float)
        bajo_c = g.loc[g["par_valido_c"], "ratio_linea_pobreza_t"].astype(float)
        fila = {"h": h, "ANO4_t": anio, "era": g["era"].iloc[0], "n_A": len(bajo_a), "n_C": len(bajo_c),
                "pct_excluido_por_imputacion": 100 * (1 - len(bajo_c) / len(bajo_a)) if len(bajo_a) else np.nan}
        for q in (10, 50, 90):
            fila[f"p{q}_A"] = bajo_a.quantile(q / 100)
            fila[f"p{q}_C"] = bajo_c.quantile(q / 100)
            fila[f"p{q}_C_menos_A"] = fila[f"p{q}_C"] - fila[f"p{q}_A"]
        filas.append(fila)
composicion = pd.DataFrame(filas)
composicion.to_csv(META / "dos_universos_composicion_c.csv", index=False)
display(composicion[composicion["h"] == 4].set_index("ANO4_t").drop(columns="h").round(3))

## 7. Subanálisis: salida de la pobreza entre los pobres (`t→t+4`)

Universo: `pobre_t == True`, `identidad_confirmada`, `ratio_linea_pobreza_th` no nulo y sin
imputación en ninguna punta (criterio C). Tasa de salida con umbrales 1,0/1,1/1,2, por universo
y partición.

In [ ]:
filas = []
for nombre_u, en_u in UNIVERSOS.items():
    df = pares[4][en_u(pares[4])]
    pobres = df[(df["pobre_t"] == True) & (df["identidad_confirmada"] == True)  # noqa: E712
                & df["ratio_linea_pobreza_th"].notna() & df["par_valido_c"]]
    for particion in ("todas", "train", "holdout"):
        g = pobres if particion == "todas" else pobres[pobres["particion"] == particion]
        for umbral in (1.0, 1.1, 1.2):
            filas.append({"universo": nombre_u, "particion": particion, "umbral": umbral, "n": len(g),
                          "tasa_salida": (g["ratio_linea_pobreza_th"].astype(float) >= umbral).mean()})
salida_pobreza = pd.DataFrame(filas)
salida_pobreza.to_csv(META / "dos_universos_salida_pobreza.csv", index=False)
display(salida_pobreza.pivot_table(index=["particion", "umbral"], columns="universo", values=["n", "tasa_salida"]).round(4))

## 8. Cierre — A contra C en el universo (a): ¿los imputados empeoran la predicción?

El holdout (origen `>= 2023T1`) es 100% regular, así que ahí `par_valido_c == par_valido`: A y
C **difieren solo en el entrenamiento** (los pares históricos con imputación entran o no a train).
Se evalúa la regresión a la media por decil entrenada con C contra entrenada con A, sobre el
mismo holdout.

In [ ]:
for h in (1, 4):
    holdout = pares[h][pares[h]["particion"] == "holdout"]
    assert (holdout["era"] == "regional_2016").all()
    assert (holdout["par_valido_c"] == holdout["par_valido"]).all()
    assert (holdout["par_valido_log_c"] == holdout["par_valido_log"]).all()
print("holdout 100% regular, par_valido_c == par_valido en holdout (h=1 y h=4)")

a_contra_c = linea_base[(linea_base["universo"] == "a_2003_2025") & (linea_base["modelo"] == "regresion_media_decil")]
a_contra_c = a_contra_c.pivot_table(index=["objetivo", "h", "ponderacion"], columns="criterio",
                                    values=["n_train", "mae", "rmse", "r2_vs_media_train"])
for metrica in ("mae", "rmse", "r2_vs_media_train"):
    a_contra_c[(metrica, "C_menos_A")] = a_contra_c[(metrica, "C")] - a_contra_c[(metrica, "A")]
a_contra_c = a_contra_c.sort_index(axis=1)
a_contra_c.to_csv(META / "dos_universos_holdout_a_contra_c.csv")
display(a_contra_c.round(5))


## Lectura de los resultados [confirmado sobre las tablas de arriba]

- **Conteos**: con C, el universo (a) entrena con 338.427 pares válidos (h=1) y 266.316 (h=4),
  contra 126.015 y 98.100 del universo (b), unas 2,7 veces más. Bajo A serían 430.777 y 341.822.
  Embargo y holdout son idénticos en los dos universos.
- **Línea base** (criterio C, sin ponderar, holdout común):
  - La persistencia no le gana a la media global (R² ≈ 0; en h=4 con (a) es levemente negativo).
  - La regresión a la media por decil es la única línea base con poder: R² 0,107 (a) contra
    0,101 (b) en `variacion_simetrica` h=1, y 0,132 (a) contra 0,142 (b) en h=4. Sumar 2003-2015
    mejora un poco h=1 y empeora un poco h=4. Las diferencias de MAE son de 0,002-0,005.
  - Con `peso_normalizado_trimestre` el orden es el mismo.
- **A contra C sobre el holdout** (universo (a), regresión a la media por decil): las diferencias
  son ≤0,002 en MAE y ≤0,004 en R², con signo mixto. C mejora levemente h=1 y empeora levemente
  h=4. **Para esta línea base, los pares imputados en train no cambian de forma material la
  predicción del holdout.** La justificación de C sigue siendo la de `05d`: con imputación la
  dispersión y la regresión a la media de la variable objetivo son mayores. No es una ganancia
  de predicción con esta línea base.
- **Dispersión por era**: bajo C, la histórica queda cerca de la regular (desvío 0,528 contra
  0,489 en h=1 y 0,554 contra 0,526 en h=4; IQR 0,524 contra 0,528 y 0,568 contra 0,585). Bajo A
  la histórica es claramente más dispersa (0,566 y 0,586).
- **Sensibilidad sin marcados**: (s1) quita ~10-19% de train en (a) y (s2) ~53-57%. Las métricas
  de holdout cambian <0,003 en MAE y <0,004 en R²: los cortes marcados no mueven la línea base.
- **Composición de C**: excluye 24-36% de los pares válidos por año en la era histórica (sube
  con la imputación: ~24% en 2005-2006 contra ~35-36% en 2010-2014). Corre p50 de
  `ratio_linea_pobreza_t` entre −0,07 y −0,11 y p90 entre −0,09 y −0,21. El universo histórico
  bajo C es algo más pobre que bajo A, y el corrimiento crece con la imputación. Las
  comparaciones de nivel entre eras deben tenerlo presente.
- **Salida de la pobreza** (h=4, umbral 1,0): 30,1% en (a) contra 31,4% en (b) sobre todos los
  pares; 37,7% en el holdout común.
- **Anomalía**: 2013T2 con h=4 no tiene pares (ruptura de `CODUSU`, ver `06c`), así que no aparece
  en las tablas por origen.
